# S04E02 — Wind Power

**Zadanie:** Zaprogramować harmonogram turbiny wiatrowej — zabezpieczyć podczas wichury,  
ustawić produkcję gdy wiatr jest optymalny. Limit czasu: **40 sekund**.

**Kluczowe akcje API:** `help` → `start` → raporty asynchroniczne (`getResult`) →  
`unlockCodeGenerator` → `config` → `turbinecheck` → `done`

**Techniki:** równoległe wywołania asyncio, analiza prognozy pogody, podpisywanie konfiguracji

In [ ]:
import os, json, time, asyncio, hashlib, requests
from anthropic import Anthropic
from dotenv import load_dotenv, find_dotenv
import concurrent.futures

load_dotenv(find_dotenv())

API_KEY    = os.getenv("AI_DEVS_API_KEY")
client     = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
VERIFY_URL = "https://hub.ag3nts.org/verify"
TASK       = "windpower"


def call_api(answer: dict) -> dict:
    resp = requests.post(VERIFY_URL, json={"apikey": API_KEY, "task": TASK, "answer": answer})
    return resp.json()


print("Konfiguracja OK.")

In [ ]:
# Krok 1: Pobierz dokumentację API
help_resp = call_api({"action": "help"})
print("Dokumentacja API:")
print(json.dumps(help_resp, ensure_ascii=False, indent=2))

In [ ]:
# Krok 2: Otwórz okno serwisowe
start_resp = call_api({"action": "start"})
print("Start:", json.dumps(start_resp, ensure_ascii=False, indent=2))
START_TIME = time.time()
print(f"\nTimer: okno 40s otwarte o {time.strftime('%H:%M:%S')}")

In [ ]:
# Krok 3: Kolejkuj wszystkie raporty równolegle (mamy tylko 40s!)
# Typowe raporty: weather_forecast, turbine_info, power_requirements
# Najpierw wywołaj 'help' żeby sprawdzić dokładne nazwy

def queue_report(action_name: str) -> dict:
    """Zakolejkuj raport asynchroniczny."""
    return call_api({"action": action_name})


def get_result(task_id: str, max_wait: float = 15.0) -> dict:
    """Pobieraj wynik dopóki nie jest gotowy lub nie minie limit."""
    deadline = time.time() + max_wait
    while time.time() < deadline:
        result = call_api({"action": "getResult", "taskId": task_id})
        if result.get("status") == "ready" or result.get("result"):
            return result
        time.sleep(0.5)
    return {"error": "timeout"}


# Kolejkuj raporty równolegle
report_actions = ["weatherForecast", "turbineInfo", "powerRequirements"]
task_ids = {}

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = {executor.submit(queue_report, action): action for action in report_actions}
    for future in concurrent.futures.as_completed(futures):
        action = futures[future]
        resp   = future.result()
        task_id = resp.get("taskId") or resp.get("id")
        if task_id:
            task_ids[action] = task_id
            print(f"  Zakolejkowano {action}: taskId={task_id}")
        else:
            print(f"  {action}: {resp}")

print(f"\nZakolejkowano {len(task_ids)} raportów. Czas od startu: {time.time()-START_TIME:.1f}s")

In [ ]:
# Pobieraj wyniki równolegle
reports = {}

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = {executor.submit(get_result, tid): action for action, tid in task_ids.items()}
    for future in concurrent.futures.as_completed(futures):
        action = futures[future]
        result = future.result()
        reports[action] = result
        print(f"  [{action}]: {str(result)[:200]}")

print(f"\nPobrano {len(reports)} raportów. Czas od startu: {time.time()-START_TIME:.1f}s")

In [ ]:
# Krok 4: Claude analizuje dane i planuje konfigurację
analysis_resp = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=2000,
    messages=[{
        "role": "user",
        "content": f"""Przeanalizuj dane turbiny wiatrowej i zaplanuj konfigurację harmonogramu.

Dane raportów:
{json.dumps(reports, ensure_ascii=False, indent=2)}

Zasady:
- Wichura (wiatr > max wytrzymałości turbiny) → pitchAngle ustawiony na 90°, turbineMode="protection"
- Produkcja prądu → pitchAngle optymalny wg warunków, turbineMode="production"
- Godziny zawsze z minutami i sekundami na zero: "HH:00:00"
- Każda konfiguracja wymaga unlockCode (wygeneruj przez API unlockCodeGenerator)

Zwróć JSON z planem:
{{
  "configs": {{
    "YYYY-MM-DD HH:00:00": {{"pitchAngle": X, "turbineMode": "..."}},
    ...
  }},
  "reasoning": "krótkie uzasadnienie"
}}"""
    }]
)

plan_text = analysis_resp.content[0].text
print("Plan agenta:")
print(plan_text)

try:
    start = plan_text.find("{")
    end   = plan_text.rfind("}") + 1
    plan  = json.loads(plan_text[start:end])
    configs = plan.get("configs", {})
except Exception as e:
    print(f"Błąd parsowania planu: {e}")
    configs = {}

print(f"\nZaplanowano {len(configs)} konfiguracji. Czas: {time.time()-START_TIME:.1f}s")

In [ ]:
# Krok 5: Wygeneruj kody unlock dla każdej konfiguracji
def get_unlock_code(datetime_str: str, pitch: int, mode: str) -> str:
    resp = call_api({
        "action": "unlockCodeGenerator",
        "startDate": datetime_str.split(" ")[0],
        "startHour": datetime_str.split(" ")[1] if " " in datetime_str else "00:00:00",
        "pitchAngle": pitch,
        "turbineMode": mode
    })
    return resp.get("unlockCode") or resp.get("code") or str(resp)


signed_configs = {}
for dt, cfg in configs.items():
    code = get_unlock_code(dt, cfg["pitchAngle"], cfg["turbineMode"])
    signed_configs[dt] = {**cfg, "unlockCode": code}
    print(f"  {dt}: pitchAngle={cfg['pitchAngle']}, mode={cfg['turbineMode']}, code={code[:20]}...")

print(f"\nPodpisano {len(signed_configs)} konfiguracji. Czas: {time.time()-START_TIME:.1f}s")

In [ ]:
# Krok 6: Wyślij konfigurację, sprawdź turbinę, wyślij 'done'
config_resp = call_api({"action": "config", "configs": signed_configs})
print("Config:", config_resp)
print(f"Czas: {time.time()-START_TIME:.1f}s")

# Turbine check
check_resp = call_api({"action": "turbinecheck"})
print("Turbine check:", check_resp)

# Poczekaj na wynik testu jeśli asynchroniczny
check_id = check_resp.get("taskId") or check_resp.get("id")
if check_id:
    check_result = get_result(check_id, max_wait=10)
    print("Check result:", check_result)

# Done
done_resp = call_api({"action": "done"})
print("\nDone:", done_resp)
print(f"Czas łączny: {time.time()-START_TIME:.1f}s (limit: 40s)")

import re
if "FLG:" in str(done_resp):
    flags = re.findall(r'\{FLG:[^}]+\}', str(done_resp))
    print(f"\nFLAGA: {flags}")